In [2]:
import librosa
import numpy as np

def extract_features(file_path):
    y, sr = librosa.load(file_path, duration=3)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc.T, axis=0)

    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    spectral_rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))

    return np.hstack([mfcc_mean, zcr, spectral_centroid, spectral_rolloff])


In [3]:
import os
import pandas as pd

real_path = "dataset/real"
fake_path = "dataset/fake"

X = []
y = []

for file in os.listdir(real_path):
    if file.endswith(".wav"):
        X.append(extract_features(os.path.join(real_path, file)))
        y.append(0)   # 0 = real

for file in os.listdir(fake_path):
    if file.endswith(".wav"):
        X.append(extract_features(os.path.join(fake_path, file)))
        y.append(1)   # 1 = fake

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)


Dataset shape: (42, 16)


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [5]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)


,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [6]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00         5

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9

[[4 0]
 [0 5]]


In [8]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

svm_model = make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True))
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))


SVM Accuracy: 0.8888888888888888


In [9]:
 
import joblib

joblib.dump(svm_model, "voice_detector.pkl")
print("Model saved!")


Model saved!


In [11]:
loaded_model = joblib.load("voice_detector.pkl")

test_file = "r1.wave"
features = extract_features(test_file).reshape(1, -1)

prediction = loaded_model.predict(features)[0]

if prediction == 0:
    print("✅ Real Human Voice")
else:
    print("❌ Synthetic / Fake Voice")


/var/folders/j9/nq0w1x9j0v56chqn6dthhnzh0000gn/T/ipykernel_850/3849008333.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, duration=3)
/opt/anaconda3/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


FileNotFoundError: [Errno 2] No such file or directory: 'r1.wave'

In [17]:
import os
print(os.getcwd())


/Users/nishanthmekala/Minor Project


In [1]:
!pip install librosa numpy pandas matplotlib scikit-learn soundfile

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 1.8 MB/s  0:00:00

   ----------------- ---------------------- 4/9 [standard-aifc]
   -------------------------- ------------- 6/9 [pooch]
   ------------------------------- -------- 7/9 [audioread]
   ----------------------------------- ---- 8/9 [librosa]
   ----------------------------------- ---- 8/9 [librosa]
   ----------------------------------- ---- 8/9 [librosa]
   ----------------------------------- ---- 8/9 [librosa]
   ---------------------------------------- 9/9 [librosa]

